# <font color="#418FDE" size="6.5" uppercase>**Gaussian Processes**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Explain Gaussian process predictions in terms of kernels, mean estimates, and uncertainty intervals. 
- Fit Gaussian Process Regression and Classification models with appropriate kernel choices. 
- Evaluate kernel hyperparameters, alpha noise settings, and computational scaling constraints. 


## **1. GP Predictions**

### **1.1. Kernel Based Predictions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_01.jpg?v=1783846138" width="250">



>* Kernels weight predictions by input similarity.
>* They encode patterns without fixed equations.

>* Kernels define similarity beyond physical distance.
>* Good kernels transfer information using meaningful patterns.

>* Dense regions enable stronger, evidence-based predictions.
>* Sparse regions require caution and weaker sharing.



In [ ]:
#@title Python Code - Kernel Based Predictions

# Kernels weight nearby observations strongly.
# Predictions include mean and uncertainty.
# This example visualizes local influence.

import numpy as np
import matplotlib.pyplot as plt

# Use a deterministic random seed.
rng = np.random.default_rng(7)

# Create small one dimensional training data.
X_train = np.array([-4.0, -2.0, -0.5, 1.0, 3.0])
y_train = np.sin(X_train) + rng.normal(0.0, 0.12, 5)

# Create test locations for prediction.
X_test = np.linspace(-5.0, 5.0, 300)

# Define a smooth radial kernel.
def rbf_kernel(xa, xb, length_scale=1.2, variance=1.0):
    sqdist = (xa[:, None] - xb[None, :]) ** 2
    return variance * np.exp(-0.5 * sqdist / length_scale**2)

# Build kernel matrices safely.
K = rbf_kernel(X_train, X_train) + 0.08 * np.eye(len(X_train))
Ks = rbf_kernel(X_train, X_test)
Kss = rbf_kernel(X_test, X_test)

# Validate matrix shapes before solving.
assert K.shape == (5, 5)
assert Ks.shape[0] == len(X_train)

# Compute predictive mean and covariance.
alpha = np.linalg.solve(K, y_train)
pred_mean = Ks.T @ alpha
v = np.linalg.solve(K, Ks)
pred_cov = Kss - Ks.T @ v

# Extract stable predictive standard deviations.
pred_var = np.clip(np.diag(pred_cov), 0.0, None)
pred_std = np.sqrt(pred_var)

# Compare influence at two test points.
near_x = np.array([1.2])
far_x = np.array([4.8])
near_weights = rbf_kernel(X_train, near_x).ravel()
far_weights = rbf_kernel(X_train, far_x).ravel()

# Print a short teaching summary.
print("Nearest-region uncertainty:", round(pred_std[np.argmin(np.abs(X_test - 1.2))], 3))
print("Far-region uncertainty:", round(pred_std[np.argmin(np.abs(X_test - 4.8))], 3))
print("Near-point strongest weight:", round(near_weights.max(), 3))
print("Far-point strongest weight:", round(far_weights.max(), 3))

# Plot mean, interval, and observations.
plt.figure(figsize=(8, 4.8))
plt.fill_between(
    X_test,
    pred_mean - 1.96 * pred_std,
    pred_mean + 1.96 * pred_std,
    alpha=0.25,
    label="95% interval",
)
plt.plot(X_test, pred_mean, label="Predictive mean")
plt.scatter(X_train, y_train, color="black", zorder=3, label="Observed points")

# Mark two example prediction locations.
plt.axvline(1.2, color="green", linestyle="--", alpha=0.7)
plt.axvline(4.8, color="red", linestyle="--", alpha=0.7)
plt.title("Kernel Based Gaussian Process Predictions")
plt.xlabel("Input location")
plt.ylabel("Predicted value")
plt.legend()

# Show the final teaching figure.
plt.tight_layout()
plt.show()



### **1.2. Posterior Mean**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_02.jpg?v=1783846140" width="250">



>* Posterior mean updates prior using observed data.
>* Kernel similarity guides predictions toward nearby patterns.

>* Predictions weight observations by kernel relationships.
>* They balance redundancy and adapt smoothness.

>* Posterior mean gives the central prediction.
>* Trust depends on data, kernel, uncertainty.



In [ ]:
#@title Python Code - Posterior Mean

# Posterior mean uses weighted training information.
# Nearby points influence predictions more strongly.
# This script visualizes smooth posterior mean behavior.

import numpy as np
import matplotlib.pyplot as plt

# Use a deterministic seed.
rng = np.random.default_rng(7)

# Create tiny one dimensional training data.
X_train = np.array([[-2.0], [-0.8], [0.2], [1.4], [2.2]])
y_train = np.sin(X_train[:, 0]) + rng.normal(0, 0.08, 5)

# Create test inputs for prediction.
X_test = np.linspace(-3, 3, 300).reshape(-1, 1)

# Define a simple radial basis kernel.
def rbf_kernel(a, b, length_scale=0.9, variance=1.0):
    sqdist = (a - b.T) ** 2
    return variance * np.exp(-0.5 * sqdist / length_scale**2)

# Build kernel matrices safely.
K = rbf_kernel(X_train, X_train) + 0.05 * np.eye(len(X_train))
Ks = rbf_kernel(X_test, X_train)

# Check matrix dimensions before solving.
assert K.shape == (5, 5)
assert Ks.shape[1] == len(X_train)

# Compute posterior mean weights.
alpha = np.linalg.solve(K, y_train)

# Compute posterior mean predictions.
posterior_mean = Ks @ alpha

# Summarize the weighted prediction idea.
print("Posterior mean combines observed outputs with kernel weights.")
print(f"Training points: {len(X_train)}, test points: {len(X_test)}")
print(f"Mean near x=0: {posterior_mean[np.argmin(np.abs(X_test[:,0]))]:.3f}")

# Plot training data and posterior mean.
plt.figure(figsize=(8, 4))
plt.scatter(X_train[:, 0], y_train, color="black", label="training points")
plt.plot(X_test[:, 0], posterior_mean, color="royalblue", label="posterior mean")
plt.axhline(0.0, color="gray", linestyle="--", linewidth=1, label="zero prior mean")

# Add labels for interpretation.
plt.title("Gaussian Process Posterior Mean")
plt.xlabel("Input x")
plt.ylabel("Predicted function value")
plt.legend()

# Show the final teaching plot.
plt.tight_layout()
plt.show()



### **1.3. Uncertainty Intervals**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_03.jpg?v=1783846141" width="250">



>* Intervals show prediction confidence across input locations.
>* Dense nearby data narrows uncertainty; sparse data widens.

>* Kernels shape uncertainty across nearby predictions.
>* Noise and data coverage also affect intervals.

>* Intervals show local confidence, not certainty.
>* Wide intervals suggest caution and more data.



In [ ]:
#@title Python Code - Uncertainty Intervals

# Gaussian processes express predictions as distributions.
# Uncertainty bands widen far from observations.
# This example visualizes mean and intervals.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic seed.
rng = np.random.default_rng(7)

# Create small training data.
X_train = np.array([-4.0, -2.0, -0.5, 1.0, 3.0])
y_train = np.sin(X_train) + rng.normal(0.0, 0.12, 5)

# Create prediction locations.
X_test = np.linspace(-6.0, 6.0, 300)

# Define a smooth RBF kernel.
def rbf_kernel(xa, xb, length_scale, variance):
    xa = np.atleast_2d(xa).T
    xb = np.atleast_2d(xb).T
    sqdist = (xa - xb.T) ** 2
    return variance * np.exp(-0.5 * sqdist / length_scale**2)

# Choose kernel and noise settings.
length_scale = 1.2
variance = 1.0
noise_std = 0.18
noise_var = noise_std**2

# Build covariance matrices.
K_xx = rbf_kernel(X_train, X_train, length_scale, variance)
K_xs = rbf_kernel(X_train, X_test, length_scale, variance)
K_ss = rbf_kernel(X_test, X_test, length_scale, variance)

# Add observation noise safely.
K_xx = K_xx + noise_var * np.eye(X_train.size)

# Validate matrix dimensions.
assert K_xx.shape == (5, 5)
assert K_xs.shape[0] == X_train.size

# Compute posterior mean and covariance.
alpha = np.linalg.solve(K_xx, y_train)
mean_pred = K_xs.T @ alpha
v = np.linalg.solve(K_xx, K_xs)
cov_pred = K_ss - K_xs.T @ v

# Extract stable standard deviations.
var_pred = np.clip(np.diag(cov_pred), 0.0, None)
std_pred = np.sqrt(var_pred)

# Build a ninety-five percent interval.
lower = mean_pred - 1.96 * std_pred
upper = mean_pred + 1.96 * std_pred

# Summarize uncertainty at key locations.
near_idx = np.argmin(np.abs(X_test - 1.0))
far_idx = np.argmin(np.abs(X_test - 5.5))
print(f"Std near training point: {std_pred[near_idx]:.3f}")
print(f"Std far from data: {std_pred[far_idx]:.3f}")
print("Wider bands mean less confidence there.")

# Plot mean, data, and interval.
plt.figure(figsize=(9, 5))
plt.scatter(X_train, y_train, color="black", label="Training data")
plt.plot(X_test, mean_pred, color="royalblue", label="Posterior mean")
plt.fill_between(X_test, lower, upper, color="skyblue", alpha=0.35,
                 label="95% uncertainty interval")

# Add a reference function.
plt.plot(X_test, np.sin(X_test), "--", color="gray", label="True pattern")
plt.title("Gaussian Process Prediction Uncertainty")
plt.xlabel("Input location")
plt.ylabel("Predicted value")

# Finish the chart clearly.
plt.legend()
plt.tight_layout()
plt.show()



## **2. Kernel Design**

### **2.1. Regression Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_01.jpg?v=1783846136" width="250">



>* Kernels encode similarity and expected output patterns.
>* Choose kernels to match real process behavior.

>* Different kernels match different data patterns.
>* Kernels can combine trends, cycles, deviations.

>* Balance kernel flexibility, simplicity, and robustness.
>* Use domain knowledge for realistic predictions.



### **2.2. Kernel Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_02.jpg?v=1783846132" width="250">



>* Kernels encode similarity and expected data patterns.
>* Choose kernels using domain-specific behavior assumptions.

>* Kernels shape functions and class boundaries.
>* Match complexity to data and context.

>* Test kernels iteratively against real performance.
>* Use composite kernels, but avoid overfitting.



### **2.3. Kernel Hyperparameters**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_03.jpg?v=1783846134" width="250">



>* Hyperparameters shape similarity, smoothness, and variation.
>* They affect regression curves and classification boundaries.

>* Poor hyperparameters cause underfitting or overfitting.
>* Learn them carefully using data and context.

>* Hyperparameters reveal assumptions and feature importance.
>* Validate fits for stability, realism, scalability.



## **3. GP Scaling Constraints**

### **3.1. Kernel Hyperparameter Tuning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_01.jpg?v=1783846144" width="250">



>* Tuning kernels shapes similarity, smoothness, and predictions.
>* Choose settings matching real patterns, not overfit.

>* Tune kernels to balance flexibility and stability.
>* Check accuracy and uncertainty calibration together.

>* Blend optimization with domain knowledge.
>* Tune repeatedly for realistic, trustworthy models.



### **3.2. Kernel Hyperparameter Tuning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_02.jpg?v=1783846145" width="250">



>* Tuning kernel settings shapes predictions and uncertainty.
>* Balance sensitivity and smoothness to avoid misfit.

>* Tune kernels to match real patterns.
>* Settings express beliefs about system behavior.

>* Optimization may find different plausible hyperparameters.
>* Check fit, stability, and realistic predictions.



### **3.3. Scaling Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_03.jpg?v=1783846147" width="250">



>* GP costs rise sharply with data size.
>* Large datasets can make GP impractical.

>* Repeated tuning makes Gaussian processes much slower.
>* Scaling limits force simpler, practical modeling choices.

>* GPs suit modest data, not all problems.
>* Large datasets need approximations for practical use.



In [ ]:
#@title Python Code - Scaling Limits

# This example shows Gaussian process scaling limits.
# We compare matrix costs as data grows.
# Small sizes keep the lesson fast.

import time
import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Define small dataset sizes.
sizes = np.array([40, 80, 120, 160])

# Store timing and memory summaries.
fit_times, pred_times, memories = [], [], []

# Build a simple radial kernel.
def rbf_kernel(x1, x2, length=0.25):
    d2 = (x1 - x2.T) ** 2
    return np.exp(-0.5 * d2 / length**2)

# Loop through increasing training sizes.
for n in sizes:
    x_train = np.linspace(0.0, 1.0, n)[:, None]
    y_train = np.sin(2 * np.pi * x_train[:, 0])

    # Add tiny noise for stability.
    y_train = y_train + 0.05 * rng.normal(size=n)

    # Build covariance and estimate memory.
    start_fit = time.perf_counter()
    k_xx = rbf_kernel(x_train, x_train) + 1e-6 * np.eye(n)
    chol = np.linalg.cholesky(k_xx)

    # Solve for GP mean weights.
    alpha = np.linalg.solve(chol.T, np.linalg.solve(chol, y_train))
    fit_times.append(time.perf_counter() - start_fit)

    # Predict on a fixed grid.
    x_test = np.linspace(0.0, 1.0, 120)[:, None]
    start_pred = time.perf_counter()
    k_tx = rbf_kernel(x_test, x_train)
    mean_pred = k_tx @ alpha
    pred_times.append(time.perf_counter() - start_pred)

    # Save covariance memory in megabytes.
    memories.append(k_xx.nbytes / 1024**2)

# Print a short scaling summary.
for n, ft, pt, mem in zip(
    sizes, fit_times, pred_times, memories
):
    print(
        f"n={n:3d} | fit={ft*1000:6.2f} ms | ",
        f"predict={pt*1000:6.2f} ms | K={mem:5.3f} MB",
        sep=""
    )

# Plot how fit time changes.
plt.figure(figsize=(6, 4))
plt.plot(sizes, fit_times, marker="o", label="fit time")
plt.plot(sizes, pred_times, marker="s", label="predict time")

# Label the teaching plot.
plt.xlabel("Training points")
plt.ylabel("Seconds")
plt.title("Gaussian process scaling gets expensive")
plt.legend()
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Gaussian Processes**</font>


In this lecture, you learned to:
- Explain Gaussian process predictions in terms of kernels, mean estimates, and uncertainty intervals. 
- Fit Gaussian Process Regression and Classification models with appropriate kernel choices. 
- Evaluate kernel hyperparameters, alpha noise settings, and computational scaling constraints. 

In the next Lecture (Lecture B), we will go over 'Cross Decomposition'